## Let's learn OOP by practicing examples.

#### 0. A class definition for Circle

In [ ]:
import math

class Circle(object,metaclass=type):#생략했을때 들어가는 내용들
  nCircles = 0
  
  # 1) Use *args, **kargs for receiving any argument.
  # 2) super() returns a proxy object that is used to delegate method resolution 
  #    to the next class in the method resolution order(MRO).
  def __new__(cls, *args, **kargs): 
    print("Creating a new Circle instance")
    instance = super().__new__(cls)
    cls.nCircles += 1
    print(f"Number of Circle instances: {cls.nCircles}")
    return instance
  
  def __init__(self, x, y, r):
    self.x = x
    self.y = y
    self.r = r

  def __setattr__(self, name, value):
    print(f"Setting {name} to {value}")
    super().__setattr__(name, value)
    
  def area(self):
    return math.pi * self.r ** 2

#### 1. Creation

In [41]:
crc1 = Circle(0, 1, 3)

Creating a new Circle instance
Number of Circle instances: 1
Setting x to 0
Setting y to 1
Setting r to 3


In [42]:
crc2 = Circle.__new__(Circle)
Circle.__init__(crc2, 1, 2, 3)

Creating a new Circle instance
Number of Circle instances: 2
Setting x to 1
Setting y to 2
Setting r to 3


In [43]:
crc3 = Circle.__new__(Circle)
Circle.__setattr__(crc3, 'x', 0)
Circle.__setattr__(crc3, 'y', 0)
Circle.__setattr__(crc3, 'r', 1)
Circle.area(crc3)

Creating a new Circle instance
Number of Circle instances: 3
Setting x to 0
Setting y to 0
Setting r to 1


3.141592653589793

#### 2. Where are the attributes and methods of objects?  
 -    attributes of instance object ?&emsp; instance.\_\_dict__ 
 -    attributes and methods of class object ?&emsp;class.\_\_dict__

In [44]:
# Let's create a circle instance, crc.
crc = Circle(10, 10, 10)

Creating a new Circle instance
Number of Circle instances: 4
Setting x to 10
Setting y to 10
Setting r to 10


In [45]:
# attributes of instance object lie in the instance.__dict__
print(crc.__dict__)

{'x': 10, 'y': 10, 'r': 10}


In [46]:
# attributes and methods of class object lie in the Circle.__dict__.
print(Circle.__dict__)

{'__module__': '__main__', 'nCircles': 4, '__new__': <staticmethod(<function Circle.__new__ at 0x1295ed440>)>, '__init__': <function Circle.__init__ at 0x1295ed4e0>, '__setattr__': <function Circle.__setattr__ at 0x1295ed620>, 'area': <function Circle.area at 0x1295ed8a0>, '__dict__': <attribute '__dict__' of 'Circle' objects>, '__weakref__': <attribute '__weakref__' of 'Circle' objects>, '__doc__': None}


In [47]:
# Let's print the __dict__ as readable
print("{")
for k, v in Circle.__dict__.items():
    print(f"\t'{k}': \t\t{v}")
print("}")

{
	'__module__': 		__main__
	'nCircles': 		4
	'__new__': 		<staticmethod(<function Circle.__new__ at 0x1295ed440>)>
	'__init__': 		<function Circle.__init__ at 0x1295ed4e0>
	'__setattr__': 		<function Circle.__setattr__ at 0x1295ed620>
	'area': 		<function Circle.area at 0x1295ed8a0>
	'__dict__': 		<attribute '__dict__' of 'Circle' objects>
	'__weakref__': 		<attribute '__weakref__' of 'Circle' objects>
	'__doc__': 		None
}


#### 3. How to associate instance object 'crc' to class object 'Circle'?  
- instance.\_\_class__ refers to the class object
- Circle, class name, itself is a reference to the class object.
- In this example, crc.\_\_class__ == Circle.

In [48]:
# instance.__class__ == class object
cls = crc.__class__
print(cls == Circle)

print('\n')
# If you want to look at the reference value, use id()
print(id(cls))
print(id(Circle))

print('\n')
# So, what are the types of cls and Circle?
print(cls)
print(Circle)

True


4772446848
4772446848


<class '__main__.Circle'>
<class '__main__.Circle'>


#### 5. How to find out the base (parent) class objects of an instance object?  
- className.\_\_bases__ : &emsp; Returns a tuple of base classes.

In [49]:
# instance.__class__.__bases__ gives you a tuple of bases.
# Note that you might have multiple bases when you define a class like: class A(B, C).
print(crc.__class__.__bases__)

# To access the first base
print(crc.__class__.__bases__[0])

# Or, you can directly use the class name
print(Circle.__bases__)

(<class 'object'>,)
<class 'object'>
(<class 'object'>,)


In [50]:
# Let's make a function that can traverse bases of a class.
# You should provide 'class reference' as an argument.
def traverse_bases(cls):
    while cls:
        yield cls
        if cls.__bases__ == ():
            break
        for p in cls.__bases__:
            cls = p
            traverse_bases(p)

In [51]:
# Let's go through our Circle.
for cls in traverse_bases(Circle):
    print(cls)
    
##### just putting some space for next print
print('\n')   

# You can pass the argument like crc.__class__
for cls in traverse_bases(crc.__class__):
    print(cls)

<class '__main__.Circle'>
<class 'object'>


<class '__main__.Circle'>
<class 'object'>


## Let's apply our knowledge about OOP to AI Programming!

#### 1. AIPParameter : Subclasses torch.Tensor for our AI programming.  
- AIPParameter overrides \_\_new(cls, data)__
- Inside the \_\_new__ method, it uses torch.Tensor.\_make_subclass(cls, data)

In [52]:
import torch
class AIPParameter(torch.Tensor):
    def __new__(cls, data):
        if not isinstance(data, torch.Tensor):
            data = torch.tensor(data)
        param = torch.Tensor._make_subclass(cls, data)
        param.requires_grad = True
        return param

In [53]:
# Let's create an instance of AIPParameter
param = AIPParameter([1., 2., 3.])
print(param)

AIPParameter([1., 2., 3.], requires_grad=True)


In [54]:
# Let's traverse the AIPParameter class object.
for cls in traverse_bases(AIPParameter):
    print(cls)
    
print('\n')

for cls in traverse_bases(param.__class__):
    print(cls)

<class '__main__.AIPParameter'>
<class 'torch.Tensor'>
<class 'torch._C.TensorBase'>
<class 'object'>


<class '__main__.AIPParameter'>
<class 'torch.Tensor'>
<class 'torch._C.TensorBase'>
<class 'object'>


#### 2. AIPModule : Overrides \_\_setattr__(self, name, value)    
- \_\_setattr__ tracks AIPParameter and AIPModule instance objects in its own dictionaries
- Dictionaries: &emsp; \_parameters &emsp; and &emsp; \_modules 

In [55]:
class AIPModule():
    def __init__(self):
        self._parameters = {}
        self._modules = {}
    def __setattr__(self, name, value):
        if isinstance(value, AIPParameter):
            self._parameters[name] = value
        elif isinstance(value, AIPModule):
            self._modules[name] = value
        super().__setattr__(name, value)
        
    def parameters(self):
        for param in self._parameters.values():
            yield param
        for module in self._modules.values():
            yield from module.parameters()

    def named_parameters(self, prefix=''):
        for name, param in self._parameters.items():
            yield prefix + name, param
        for module_name, module in self._modules.items():
            sub_prefix = f"{prefix}{module_name}."
            yield from module.named_parameters(prefix=sub_prefix)

In [56]:
# Create an AIPModule
model = AIPModule()

# Add AIPParameters
model.param1 = AIPParameter([1., 2.])
model.param2 = AIPParameter(torch.randint(0, 10, [2, 2]).float())

# 1) Let's find out the parameters in instance object, model
print(model.__dict__)

# 2) Let's find out the parameters using instance method, parameters
print('\n')
for name, param in model.named_parameters():
    print(f"{name}:  {param}")


# 3) Let's traverse class object tree.
print('\n')
for cls in traverse_bases(model.__class__):
    print(cls)

{'_parameters': {'param1': AIPParameter([1., 2.], requires_grad=True), 'param2': AIPParameter([[6., 6.],
              [0., 0.]], requires_grad=True)}, '_modules': {}, 'param1': AIPParameter([1., 2.], requires_grad=True), 'param2': AIPParameter([[6., 6.],
              [0., 0.]], requires_grad=True)}


param1:  AIPParameter([1., 2.], requires_grad=True)
param2:  AIPParameter([[6., 6.],
              [0., 0.]], requires_grad=True)


<class '__main__.AIPModule'>
<class 'object'>


#### 3. Subclassing AIPModule

In [57]:
# A_Module subclasses AIPModule
class A_Module(AIPModule):
    def __init__(self):
        super().__init__()
        self.param2 = AIPParameter([4., 5, 6])
        self.param3 = AIPParameter([7., 8, 9])

# B_Module subclasses AIPModule and has a A_Module as an attribute.
class B_Module(AIPModule):
    def __init__(self):
        super().__init__()
        self.param1 = AIPParameter([1., 2, 3])
        self.AMoudle = A_Module()

In [58]:
# Let's traverse A_Module
for cls in traverse_bases(A_Module):
    print(cls)

print('\n')

# Let's traverse B_Module
for cls in traverse_bases(B_Module):
    print(cls)

<class '__main__.A_Module'>
<class '__main__.AIPModule'>
<class 'object'>


<class '__main__.B_Module'>
<class '__main__.AIPModule'>
<class 'object'>


In [59]:
# Let's findout the parameters in each module

# 1) A_Module
modelA = A_Module()
for name, param in modelA.named_parameters():
    print(f"{name}: {param}")

print('\n')

# 2) B_Module
modelB = B_Module()
for name, param in modelB.named_parameters():
    print(f"{name}: {param}")

param2: AIPParameter([4., 5., 6.], requires_grad=True)
param3: AIPParameter([7., 8., 9.], requires_grad=True)


param1: AIPParameter([1., 2., 3.], requires_grad=True)
AMoudle.param2: AIPParameter([4., 5., 6.], requires_grad=True)
AMoudle.param3: AIPParameter([7., 8., 9.], requires_grad=True)


In [60]:
# Let's compare Python __dict__
for k, v in modelB.__dict__.items():
    print(f"{k}:  {v}")

_parameters:  {'param1': AIPParameter([1., 2., 3.], requires_grad=True)}
_modules:  {'AMoudle': <__main__.A_Module object at 0x11d233b30>}
param1:  AIPParameter([1., 2., 3.], requires_grad=True)
AMoudle:  <__main__.A_Module object at 0x11d233b30>


#### 4. AIPLinear Model
- AIPLinear(in_features, out_features, bias=True)
- weight :&emsp;[out_features, in_features]
- biase : &emsp; [out_featuers]

In [61]:
class AIPLinear(AIPModule):
    def __init__(self, in_features, out_features, use_bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.use_bias = use_bias
        
        self.weight = AIPParameter(torch.randn(out_features, in_features) * (1.0 / in_features**0.5))
        
        if use_bias:
            self.bias = AIPParameter(torch.zeros(out_features))
        else:
            self.bias = None
    
    def __call__(self, x):
        output = x @ self.weight.T 
        
        if self.use_bias and self.bias is not None: 
            output = output + self.bias
        
        return output

In [62]:
aiplinear = AIPLinear(4, 2)

In [63]:
# Let's find out base object of AIPLInear == aiplinear.__class__

for cls in traverse_bases(AIPLinear):
    print(cls)

<class '__main__.AIPLinear'>
<class '__main__.AIPModule'>
<class 'object'>


In [64]:
for name, param in aiplinear.named_parameters():
    print(f"{name}: {param}\n")

weight: AIPParameter([[-0.8457,  0.0810,  0.3338,  0.0911],
              [-0.6025, -0.6228, -0.6674, -0.1832]], requires_grad=True)

bias: AIPParameter([0., 0.], requires_grad=True)



#### 5. MLP (Multi-Linear Perceptron)

In [65]:
def relu(x):
    return torch.maximum(torch.tensor(0.0), x)

In [66]:
class MLP(AIPModule):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = AIPLinear(input_dim, hidden_dim)
        self.fc2 = AIPLinear(hidden_dim, output_dim)

    def __call__(self, x):
        x = relu(self.fc1(x))
        x = self.fc2(x)
        return x
    
mlp = MLP(2, 4, 1)

In [67]:
# Let's find out base object of MLP == mlp.__class__
for cls in traverse_bases(MLP):
    print(cls)

<class '__main__.MLP'>
<class '__main__.AIPModule'>
<class 'object'>


In [68]:
for name, param in mlp.named_parameters():
    print(f"{name}: {param}")

fc1.weight: AIPParameter([[ 1.1691,  0.1894],
              [ 0.1970, -0.2841],
              [ 1.0685,  0.9367],
              [-0.4367,  0.8251]], requires_grad=True)
fc1.bias: AIPParameter([0., 0., 0., 0.], requires_grad=True)
fc2.weight: AIPParameter([[-0.1811,  0.0766, -0.3198,  0.5396]], requires_grad=True)
fc2.bias: AIPParameter([0.], requires_grad=True)


## Demistyfy object and type in Python

#### type
- type creates a class object when you type in the class definition
- Even built-in object is an instance of type.

#### object
- Every class object is a subclass of built-in object.
- So, object is a base object for every object.
- Even type is an instance object of built-in object.

#### Every object is an instance of type and object, too.

##### 🔹Let's define a function to traverse class for a given object

In [69]:
# traverse class of an object
def traverse_class(cls):
    while cls != type:
        yield cls
        cls = cls.__class__
    yield cls
    
# traverse bases of an object.
def traverse_bases(obj):
    if not hasattr(obj, '__bases__'): yield obj
    cls = obj.__class__ if not hasattr(obj, '__bases__') else obj
    while cls:
        yield cls
        if cls.__bases__ == ():
            break
        for p in cls.__bases__:
            cls = p
            traverse_bases(cls)

In [70]:
linear = AIPLinear(4, 4)
for cls in traverse_class(linear):
    print(cls)
    
print("\n")

for cls in traverse_bases(linear):
    print(cls)

<class '__main__.AIPLinear'>
<class 'type'>


<class '__main__.AIPLinear'>
<class '__main__.AIPModule'>
<class 'object'>


In [71]:
print(object.__class__)
print(object.__bases__)

<class 'type'>
()


In [72]:
print(type.__class__)
print(type.__bases__)

<class 'type'>
(<class 'object'>,)


In [73]:
print(isinstance(object, type))
print(isinstance(type, object))

True
True


#### Very Simple Example Circle

In [74]:
crc = Circle(1, 2, 3)

Creating a new Circle instance
Number of Circle instances: 5
Setting x to 1
Setting y to 2
Setting r to 3


In [75]:
for cls in traverse_class(crc):
    print(cls)
    
print("\n")

for cls in traverse_bases(crc):
    print(cls)

<class '__main__.Circle'>
<class 'type'>


<class '__main__.Circle'>
<class 'object'>


![diagram](python_class_creation_diagram.png)


class MyClass: ...

is euqivalent to the following

In [76]:
MyClass = type.__new__(type, "MyClass", (object,), {})
type.__init__(MyClass, "MyClass", (object,), {})

In [77]:
crc1 = MyClass()

#### Class Object Creation Order
1. Let's take our AIP case as an example.
2. type and built-in object are ready for our code.
3. AIPModule class object first created by type.
4. AIPModule's base class is associated with built-in object.
5. AIPModule does its own \_\_init__.

 From this point onward, many sub-modules can be defined.

6. AIPLinear class object then create by type.
7. AIPLinear's base class is associated with AIPModule that was alredy created.
8. AIPLinear does its own \_\_init__.